# 01 - YOLO26 pose + ByteTrack exploration

Goal: see what `yolo26n-pose` gives us on a sample video before building `app/extract`.
Checks: keypoints, track IDs, torso angle, knee angle, visible-keypoint ratio (occlusion signal).

Colab: set runtime to T4 GPU. Local: run from the repo root or from `notebooks/`.

In [ ]:
# Colab only: uncomment
# !pip install -q ultralytics
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

In [ ]:
# --- config ---
MODEL_NAME = "yolo26n-pose.pt"  # fallback: "yolo11n-pose.pt"
VIDEO = Path("../data/videos/urfd/adl-01-cam0.mp4")  # change to any clip
SAMPLE_FPS = 5  # matches SAMPLE_FPS in .env
KP_CONF = 0.3  # keypoint counted as visible above this

assert VIDEO.exists(), f"Video not found: {VIDEO.resolve()}"
cap = cv2.VideoCapture(str(VIDEO))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
STRIDE = max(1, round(FPS / SAMPLE_FPS))
print(
    f"{VIDEO.name}: {W}x{H}, {FPS:.1f} fps, {N_FRAMES} frames, {N_FRAMES / FPS:.1f}s, stride={STRIDE}"
)

In [ ]:
model = YOLO(MODEL_NAME)  # downloads weights on first use
print(model.task, len(model.names), "class(es):", model.names)

## Single frame: detection + skeleton

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
cap.set(cv2.CAP_PROP_POS_FRAMES, N_FRAMES // 2)
ok, frame = cap.read()
cap.release()

res = model.predict(frame, conf=0.25, verbose=False)[0]
print("persons:", len(res.boxes))
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## Whole video: tracking + per-frame features

COCO keypoints: 5/6 shoulders, 11/12 hips, 13/14 knees, 15/16 ankles.
Angles here are only for exploration; the real feature code will live in `app/extract`.

In [ ]:
def angle_deg(a, b, c):
    # angle at b between segments b->a and b->c
    v1, v2 = np.array(a) - np.array(b), np.array(c) - np.array(b)
    cosang = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)
    return math.degrees(math.acos(np.clip(cosang, -1, 1)))


def torso_angle_from_vertical(kp):
    # 0 deg = upright, 90 deg = horizontal (lying)
    sh, hip = kp[[5, 6], :2].mean(0), kp[[11, 12], :2].mean(0)
    d = sh - hip
    return math.degrees(math.atan2(abs(d[0]), abs(d[1]) + 1e-9))


rows, annotated = [], []
stream = model.track(
    source=str(VIDEO),
    stream=True,
    persist=True,
    tracker="bytetrack.yaml",
    conf=0.25,
    vid_stride=STRIDE,
    verbose=False,
)
for i, r in enumerate(stream):
    t = i * STRIDE / FPS
    annotated.append(r.plot())
    if r.boxes is None or len(r.boxes) == 0:
        rows.append({"t": t, "n_persons": 0})
        continue
    ids = r.boxes.id.int().tolist() if r.boxes.id is not None else [None] * len(r.boxes)
    for j, tid in enumerate(ids):
        kp = r.keypoints.data[j].cpu().numpy()  # (17, 3): x, y, conf
        x1, y1, x2, y2 = r.boxes.xyxy[j].tolist()
        good = kp[:, 2] > KP_CONF
        row = {
            "t": t,
            "n_persons": len(ids),
            "track_id": tid,
            "det_conf": float(r.boxes.conf[j]),
            "bbox_h": y2 - y1,
            "visible_ratio": float(good.mean()),
        }
        if good[[5, 6, 11, 12]].all():
            row["torso_angle"] = torso_angle_from_vertical(kp)
            row["hip_x"], row["hip_y"] = kp[[11, 12], :2].mean(0)
        if good[[11, 13, 15]].all():
            row["knee_angle"] = angle_deg(kp[11, :2], kp[13, :2], kp[15, :2])
        rows.append(row)

df = pd.DataFrame(rows)
print(len(annotated), "sampled frames;", df["track_id"].nunique(), "track id(s)")
df.head()

## Plots: what the state classifier will see

In [ ]:
p = df.dropna(subset=["track_id"])
fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for tid, g in p.groupby("track_id"):
    ax[0].plot(g.t, g.torso_angle, ".-", label=f"id {tid}")
    ax[1].plot(g.t, g.knee_angle, ".-")
    ax[2].plot(g.t, g.visible_ratio, ".-")
    ax[3].plot(g.t, g.hip_y, ".-")
ax[0].set_ylabel("torso angle\n(0=upright, 90=lying)")
ax[1].set_ylabel("knee angle\n(180=straight)")
ax[2].set_ylabel("visible kp ratio")
ax[3].set_ylabel("hip y (px)")
ax[3].invert_yaxis()
ax[3].set_xlabel("time (s)")
ax[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Contact sheet of 8 evenly spaced annotated frames
idx = np.linspace(0, len(annotated) - 1, 8).astype(int)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for a, k in zip(axes.ravel(), idx):
    a.imshow(cv2.cvtColor(annotated[k], cv2.COLOR_BGR2RGB))
    a.set_title(f"t={k * STRIDE / FPS:.1f}s")
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Save the annotated clip for inspection (outputs/ is gitignored)
out = Path("../outputs/notebook_pose_demo.mp4")
out.parent.mkdir(exist_ok=True)
vw = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), SAMPLE_FPS, (W, H))
for f in annotated:
    vw.write(f)
vw.release()
print("saved", out.resolve())

## Things to note for `app/extract`
- Does the track ID stay stable? Any ID switches when the person lies down or is occluded?
- Where does `visible_ratio` drop (blankets, occlusion)? Those frames should become `UNKNOWN`, not a guess.
- Do torso angle and knee angle separate lying / sitting / standing for this scene?
- Frames with `n_persons > 1` are the caregiver case.